In [8]:
from datasets import load_dataset
import torch
from transformers import AutoTokenizer
from pathlib import Path
from tqdm import tqdm

In [9]:
ROOT = Path.cwd().parent

TOK_DIR = ROOT / "model" / "base"
DATA_DIR = ROOT / "preparation" / "data"

SPLIT = "train"
SPLITS = {"train": "train", "val": "validation", "test": "test"}
CONTEXT_LENGTH = 1024
SEED = 42
SHUFFLE = True

In [10]:
tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)

In [11]:
def row_in_context_size(size, context_length):
    return size <= context_length + 1

In [12]:
def encode_row(row):

    doc = row["document"]
    sum = row["summary"]

    prompt = "<|user|>" + doc + "<|end|>" + "<|assistant|>"
    response = sum + "<|end|>"

    prompt_ids = tokenizer(prompt).input_ids
    response_ids = tokenizer(response).input_ids

    return prompt_ids + response_ids, len(prompt_ids)

In [13]:
def build_split(hf_split: str, shuffle: bool, SEED: int):

    ds = load_dataset("EdinburghNLP/xsum", split=hf_split)
    if shuffle:
        ds = ds.shuffle(SEED)

    examples, skipped = [], 0

    for row in tqdm(ds):
        ids, prompt_len = encode_row(row)

        if not row_in_context_size(len(ids), CONTEXT_LENGTH):
            skipped += 1
            continue
        if len(ids) - prompt_len < 10:
            skipped += 1
            continue

        examples.append({"ids": ids, "prompt_len": prompt_len})

    return examples, skipped

In [14]:
for name, hf_split in SPLITS.items():
    out = DATA_DIR / f"{name}.pt"

    examples, skipped = build_split(hf_split, SHUFFLE, SEED)

    torch.save(
        {
            "examples": examples,
            "context_length": CONTEXT_LENGTH,
            "seed": SEED,
            "vocab_size": len(tokenizer)
        },
        out
    )

    print(f"Rows skipped: {skipped}")

100%|██████████| 204045/204045 [03:17<00:00, 1032.38it/s]


Rows skipped: 22272


100%|██████████| 11332/11332 [00:10<00:00, 1034.25it/s]


Rows skipped: 1219


100%|██████████| 11334/11334 [00:10<00:00, 1045.76it/s]


Rows skipped: 1205
